# EDA — TTC Route 39 (Finch West) · Bus Delay Prediction
**DAMO-699 Capstone Project**  
Team: Saurav · Kriti · Pooja · Shristi &nbsp;|&nbsp; Supervisor: Dr. Bilal El Toufaili  
Deadline: March 20, 2026

---
> Work through each step in order. Markdown cells describe the task; code cells are the skeleton.  
> Fill in findings as comments or new cells as you go.


## Step 0 — Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 4)

ROUTE    = 39
DATA_DIR = Path('../../data')
print(f"Route {ROUTE} EDA · data dir: {DATA_DIR.resolve()}")


## Step 1 — Understand the Data Structure
One wrong assumption here breaks all downstream analysis.

### 1A — Column Mapping
| Task | Notes |
|---|---|
| List every column and label its source | AVL/VISION · GTFS · TransSee |
| Confirm what one row represents | GPS ping every 20 s? or stop-arrival event? |
| Identify the delay column | pre-computed or calculate from actual vs scheduled? |
| Identify `trip_id`, `stop_id`, `vehicle_id`, `timestamp` | check dtypes — are timestamps datetime? |


In [ ]:
# ── 1A: Load raw data ─────────────────────────────────────────────────────
# Adjust path to match your actual saved file
# df = pd.read_parquet(DATA_DIR / 'processed/route39_stop_events.parquet')

# df.info()
# print("\\nColumns:", df.columns.tolist())
# print("\\nOne row:\\n", df.iloc[0])

# Source map (fill in after inspection):
# col_source = {
#     'trip_id':             'GTFS',
#     'stop_id':             'GTFS',
#     'vehicle_id':          'AVL/VISION',
#     'timestamp':           'AVL/VISION',
#     'delay_sec':           'computed: actual_arrival - scheduled_arrival',
#     'lat':                 'AVL/VISION',
#     'lon':                 'AVL/VISION',
#     'run_number':          'AVL/VISION (NULL = RAD bus)',
#     'scheduled_headway_sec': 'GTFS frequencies.txt',
#     'actual_headway_sec':  'TransSee',
# }
print("TODO 1A: load df and map columns")


### 1B — Date & Schedule Range
| Task | Notes |
|---|---|
| Earliest and latest date | `df['timestamp'].agg(['min','max'])` |
| Spans multiple TTC schedule board periods? | note change dates — scheduled headways may differ |
| Unique dates, trips, vehicles, stops | `df.nunique()` |


In [ ]:
# ── 1B: Date range ────────────────────────────────────────────────────────
# print("Date range:", df['timestamp'].min(), "→", df['timestamp'].max())
# print("\\nUnique counts:\\n", df[['trip_id','stop_id','vehicle_id']].nunique())
# print("Unique dates:", df['timestamp'].dt.date.nunique())

# TTC schedule board periods (approximate):
# P1: 2025-01-05 – 2025-03-29
# P2: 2025-03-30 – 2025-06-28
# P3: 2025-06-29 – 2025-09-27
# P4: 2025-09-28 – 2025-12-27
print("TODO 1B: date range")


### 1C — GPS Data Sanity
| Task | Notes |
|---|---|
| Rogue lat/lon readings | filter to Route 39 bounding box |
| NULL `run_number` records | RAD buses — count and exclude from delay analysis |
| 20-second polling interval consistency | gaps > 60 s per vehicle per trip |


In [ ]:
# ── 1C: GPS sanity ────────────────────────────────────────────────────────
# Route 39 bounding box (approx — adjust if needed):
LAT_MIN, LAT_MAX = 43.72, 43.82; LON_MIN, LON_MAX = -79.58, -79.33

# rogue = df[(df['lat'] < LAT_MIN) | (df['lat'] > LAT_MAX) |
#            (df['lon'] < LON_MIN) | (df['lon'] > LON_MAX)]
# print(f"Rogue GPS: {len(rogue):,} ({len(rogue)/len(df)*100:.2f}%)")

# rad = df[df['run_number'].isna()]
# print(f"NULL run_number (RAD buses): {len(rad):,}")

# gaps = (df.sort_values(['vehicle_id','trip_id','timestamp'])
#           .groupby(['vehicle_id','trip_id'])['timestamp']
#           .diff().dt.total_seconds())
# print(f"Gaps > 60 s: {(gaps > 60).sum():,}")
print("TODO 1C: GPS sanity")


## Step 2 — Time Dimension
Steve Munro's insight: the same route behaves very differently by time of day and day of week.

**Tag every record first:**
```python
hour_of_day   # 0–23
day_of_week   # Mon=0 … Sun=6
is_weekday    # True / False
time_bucket   # 'Early Morning' / 'AM Peak' / 'Midday' / 'PM Peak' / 'Evening' / 'Late Night'
```


In [ ]:
# ── Tag time fields ────────────────────────────────────────────────────────
def tag_time(df, ts='timestamp'):
    df = df.copy()
    df['hour_of_day'] = df[ts].dt.hour
    df['day_of_week'] = df[ts].dt.dayofweek
    df['is_weekday']  = df['day_of_week'] < 5
    bins   = [-1, 4, 6, 8, 14, 18, 21, 23]
    labels = ['Late Night','Early Morning','AM Peak','Midday','PM Peak','Evening','Late Night2']
    df['time_bucket'] = (pd.cut(df['hour_of_day'], bins=bins, labels=labels)
                           .astype(str).replace('Late Night2','Late Night'))
    return df
# df = tag_time(df)
print("TODO 2: apply tag_time()")


### 2A — Day-of-Week & Time-Bucket Comparison

In [ ]:
# ── 2A ─────────────────────────────────────────────────────────────────────
# delay_col = 'delay_sec'
# print(df.groupby('day_of_week')[delay_col].agg(['mean','std']))
# print(df.groupby('time_bucket')[delay_col].agg(['mean','std']))

# # Heatmap
# pivot = df.pivot_table(values=delay_col, index='time_bucket',
#                        columns='day_of_week', aggfunc='mean')
# sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn_r')
# plt.title(f'Route {ROUTE} — Mean Delay (s) by Day × Time Bucket')
# plt.tight_layout(); plt.show()
print("TODO 2A")


### 2B — Hour-of-Day Pattern

In [ ]:
# ── 2B ─────────────────────────────────────────────────────────────────────
# hourly = df.groupby('hour_of_day').agg(
#     mean_delay=(delay_col, 'mean'), trip_count=('trip_id','nunique')).reset_index()
# fig, (ax1, ax2) = plt.subplots(2,1, figsize=(12,6), sharex=True)
# ax1.plot(hourly['hour_of_day'], hourly['mean_delay'], marker='o')
# ax1.set_ylabel('Mean Delay (s)')
# ax2.bar(hourly['hour_of_day'], hourly['trip_count'])
# ax2.set_xlabel('Hour of Day')
# plt.tight_layout(); plt.show()
print("TODO 2B")


### 2C — Schedule Board Periods

In [ ]:
# ── 2C ─────────────────────────────────────────────────────────────────────
# periods = {1:('2025-01-05','2025-03-29'), 2:('2025-03-30','2025-06-28'),
#             3:('2025-06-29','2025-09-27'), 4:('2025-09-28','2025-12-27')}
# def assign_period(d):
#     for p,(s,e) in periods.items():
#         if pd.Timestamp(s) <= d <= pd.Timestamp(e): return p
# df['board_period'] = df['timestamp'].dt.normalize().apply(assign_period)
# print(df.groupby('board_period')[delay_col].agg(['mean','std','count']))
print("TODO 2C")


## Step 3 — Space Dimension
Delay doesn't behave the same at every stop on Route 39.

**Tag every record:**
```python
stop_sequence          # 1 = first stop from terminal (from GTFS)
distance_from_terminal # metres (from GTFS shape data)
direction              # 'Northbound' / 'Southbound'
```


In [ ]:
# ── Tag spatial fields ─────────────────────────────────────────────────────
# stop_times = pd.read_csv(DATA_DIR / 'gtfs/stop_times.txt')
# df = df.merge(stop_times[['trip_id','stop_id','stop_sequence']],
#               on=['trip_id','stop_id'], how='left')
# trips = pd.read_csv(DATA_DIR / 'gtfs/trips.txt')
# dir_map = trips.set_index('trip_id')['direction_id'].map({0:'Northbound',1:'Southbound'})
# df['direction'] = df['trip_id'].map(dir_map)
print("TODO 3: merge GTFS spatial fields")


### 3A — Delay Along the Route

In [ ]:
# ── 3A ─────────────────────────────────────────────────────────────────────
# stop_delay = df.groupby('stop_sequence')[delay_col].mean().reset_index()
# plt.plot(stop_delay['stop_sequence'], stop_delay[delay_col], marker='.')
# plt.xlabel('Stop Sequence'); plt.ylabel('Mean Delay (s)')
# plt.title(f'Route {ROUTE} — Mean Delay by Stop Sequence')
# plt.tight_layout(); plt.show()
# print("Top 5 delay stops:\\n", df.groupby('stop_id')[delay_col].mean().nlargest(5))
print("TODO 3A")


### 3B — Directional Differences (NB vs SB)

In [ ]:
# ── 3B ─────────────────────────────────────────────────────────────────────
# for direction, g in df.groupby('direction'):
#     s = g.groupby('stop_sequence')[delay_col].mean()
#     plt.plot(s.index, s.values, label=direction, marker='.')
# plt.legend(); plt.xlabel('Stop Sequence'); plt.ylabel('Mean Delay (s)')
# plt.title(f'Route {ROUTE} — Delay by Direction'); plt.tight_layout(); plt.show()
print("TODO 3B")


### 3C — Dwell Time Hotspots (TransSee)

In [ ]:
# ── 3C ─────────────────────────────────────────────────────────────────────
# ts = pd.read_parquet(DATA_DIR / f'processed/transsee_route{ROUTE}.parquet')
# dwell = ts.groupby('stop_id')['dwell_sec'].mean().nlargest(10)
# print("Top 10 dwell-time stops (s):\\n", dwell)
# stop_metrics = df.groupby('stop_id')[delay_col].mean().rename('mean_delay')
# print("Corr dwell vs delay:", stop_metrics.to_frame().join(dwell.rename('mean_dwell')).corr())
print("TODO 3C")


## Step 4 — Delay Distribution

### 4A — Overall Distribution

In [ ]:
# ── 4A ─────────────────────────────────────────────────────────────────────
# fig, (ax1, ax2) = plt.subplots(1,2, figsize=(14,4))
# ax1.hist(df[delay_col].dropna(), bins=120, edgecolor='none')
# ax1.set_xlabel('Delay (s)'); ax1.set_title(f'Route {ROUTE} — Delay Histogram')
# ax2.boxplot(df[delay_col].dropna(), vert=False)
# ax2.set_xlabel('Delay (s)'); plt.tight_layout(); plt.show()
# stats = df[delay_col].describe(percentiles=[.5,.9,.95,.99])
# print(stats)
# print(f"% early (<0):     {(df[delay_col]<0).mean()*100:.1f}%")
# print(f"% >10 min late:   {(df[delay_col]>600).mean()*100:.1f}%")
# print(f"% >15 min late:   {(df[delay_col]>900).mean()*100:.1f}%")
print("TODO 4A")


### 4B — Headway Distribution & Bunching

In [ ]:
# ── 4B ─────────────────────────────────────────────────────────────────────
# Actual headway = time between consecutive buses at same stop
# df = df.sort_values(['stop_id','timestamp'])
# df['actual_headway_sec'] = df.groupby('stop_id')['timestamp'].diff().dt.total_seconds()

# Bunching: actual_headway < 50% of scheduled
# if 'scheduled_headway_sec' in df.columns:
#     df['bunching'] = df['actual_headway_sec'] < (0.5 * df['scheduled_headway_sec'])
#     print(f"Bunching rate: {df['bunching'].mean()*100:.1f}%")

# fig, ax = plt.subplots(figsize=(12,4))
# ax.hist(df['actual_headway_sec'].dropna()/60, bins=80, label='Actual', alpha=0.7)
# ax.hist(df['scheduled_headway_sec'].dropna()/60, bins=40, label='Scheduled', alpha=0.7)
# ax.set_xlabel('Headway (min)'); ax.legend()
# plt.title(f'Route {ROUTE} — Headway Distribution'); plt.tight_layout(); plt.show()
print("TODO 4B")


### 4C — Outlier Investigation

In [ ]:
# ── 4C ─────────────────────────────────────────────────────────────────────
# top20 = df.nlargest(20, delay_col)[['timestamp','stop_id','vehicle_id','trip_id',delay_col]]
# print("Top 20 extreme delays:\\n", top20.to_string())
# print("\\nDates of extremes:", top20['timestamp'].dt.date.value_counts().head())
# DECISION: cap / log-transform / keep as-is → [fill in]
print("TODO 4C")


## Step 5 — Trip-Level Propagation

### 5A — Within-Trip Delay Progression

In [ ]:
# ── 5A ─────────────────────────────────────────────────────────────────────
# prog = (df.sort_values(['trip_id','stop_sequence'])
#           .groupby('trip_id').agg(
#               start_delay=(delay_col,'first'),
#               end_delay=(delay_col,'last'),
#               n_stops=('stop_sequence','count')).reset_index())
# prog['delay_added'] = prog['end_delay'] - prog['start_delay']
# print(f"Avg delay added per stop: {(prog['delay_added']/prog['n_stops']).mean():.1f} s")
# print(f"% trips recovering delay: {(prog['delay_added']<0).mean()*100:.1f}%")
# plt.scatter(prog['start_delay'], prog['end_delay'], alpha=0.1, s=4)
# plt.axline((0,0), slope=1, color='red', ls='--', label='no change')
# plt.xlabel('Start Delay (s)'); plt.ylabel('End Delay (s)')
# plt.title(f'Route {ROUTE} — Trip Start vs End Delay'); plt.legend(); plt.tight_layout(); plt.show()
print("TODO 5A")


### 5B — Cascading Effect & Lag Feature

In [ ]:
# ── 5B ─────────────────────────────────────────────────────────────────────
# veh_trips = (df.sort_values(['vehicle_id','timestamp'])
#               .groupby(['vehicle_id','trip_id'])
#               .agg(trip_start=('timestamp','min'), end_delay=(delay_col,'last'))
#               .reset_index().sort_values(['vehicle_id','trip_start']))
# veh_trips['prev_end_delay'] = veh_trips.groupby('vehicle_id')['end_delay'].shift(1)
# corr = veh_trips[['end_delay','prev_end_delay']].corr().iloc[0,1]
# print(f"Corr end_delay[N] vs end_delay[N-1]: {corr:.3f}")

# Create lag feature:
# prev_map = veh_trips.set_index('trip_id')['prev_end_delay']
# df['previous_trip_delay'] = df['trip_id'].map(prev_map)
print("TODO 5B — lag feature 'previous_trip_delay' is likely your strongest feature")


## Step 6 — Vehicle-Level Patterns

In [ ]:
# ── Step 6 ─────────────────────────────────────────────────────────────────
# print("Top 10 worst vehicles (mean delay):")
# print(df.groupby('vehicle_id')[delay_col].agg(['mean','std','count']).nlargest(10,'mean'))

# GPS gap count per vehicle:
# gaps_per_veh = (df.sort_values(['vehicle_id','timestamp'])
#                  .groupby('vehicle_id')['timestamp']
#                  .apply(lambda x: (x.diff().dt.total_seconds() > 60).sum()))
# print("\\nVehicles with most GPS gaps:", gaps_per_veh.nlargest(10))

# Include vehicle_id as feature?  Rule: ≥30 trips per vehicle
# eligible = (df.groupby('vehicle_id')['trip_id'].nunique() >= 30).sum()
# print(f"Vehicles with ≥30 trips: {eligible}")
print("TODO 6")


## Step 7 — Missing Data

In [ ]:
# ── Step 7 ─────────────────────────────────────────────────────────────────
# missing = df.isnull().sum().sort_values(ascending=False)
# print("Missing per column:")
# print(pd.concat([missing, missing/len(df)*100], axis=1, keys=['count','%']).head(20))

# Is missingness concentrated at peak hours?
# for col in df.columns[df.isnull().any()]:
#     pk = df[df['time_bucket'].isin(['AM Peak','PM Peak'])][col].isnull().mean()
#     op = df[~df['time_bucket'].isin(['AM Peak','PM Peak'])][col].isnull().mean()
#     print(f"{col}: peak={pk:.3f}  off-peak={op:.3f}")
print("TODO 7: missing data audit")


In [ ]:
# ── Step 7 continued ───────────────────────────────────────────────────────
# Missing by stop:
# print("Stops with highest missingness:")
# print(df.groupby('stop_id')[delay_col].apply(lambda x: x.isnull().mean()).nlargest(10))

# Trip completeness:
# trip_stops = df.groupby('trip_id')['stop_sequence'].nunique()
# expected   = df['stop_sequence'].max()
# print(f"% trips with <80% stop coverage: {(trip_stops < expected*0.8).mean()*100:.1f}%")

# Imputation strategy (document here):
# - GPS gaps ≤60 s:   forward-fill lat/lon
# - NULL run_number:  exclude (RAD buses)
# - Missing stop_id:  drop row
# - Missing delay_sec: recompute if possible, else drop
print("TODO 7 continued")


## Step 8 — Feature Readiness Check

In [ ]:
# ── Step 8: Verify each candidate feature ─────────────────────────────────
FEATURES = [
    'hour_of_day', 'day_of_week', 'time_bucket',
    'stop_sequence', 'distance_from_terminal', 'direction',
    'scheduled_headway_sec', 'actual_headway_sec', 'previous_trip_delay',
]
# for f in FEATURES:
#     if f in df.columns:
#         print(f"  {f:<30} dtype={df[f].dtype}  nulls={df[f].isnull().mean()*100:.1f}%")
#     else:
#         print(f"  {f:<30} *** MISSING ***")
print("TODO 8: verify feature readiness for Route 39")


In [ ]:
# ── Step 8: Correlation matrix ─────────────────────────────────────────────
# Encode categoricals:
# df_enc = df[FEATURES + [delay_col]].copy()
# df_enc['dir_enc']    = (df_enc['direction'] == 'Northbound').astype(int)
# df_enc['bucket_enc'] = df_enc['time_bucket'].astype('category').cat.codes

# num_feats = [f for f in FEATURES if df_enc[f].dtype in ['float64','int64','int32']]
# num_feats += ['dir_enc','bucket_enc', delay_col]
# corr = df_enc[num_feats].corr()[delay_col].sort_values(ascending=False)
# print("Correlation with delay_sec:\\n", corr)
# weak = corr[corr.abs() < 0.05].index.tolist()
# print(f"\\nWeak features (|r|<0.05): {weak}")
print("TODO 8: correlation matrix")


## Summary — Key EDA Questions to Answer

| Question | Finding |
|---|---|
| Does delay vary by time of day and day of week on Route 39? | |
| Which stops are the worst delay generators, and why? | |
| Does delay compound along the stop sequence, or recover? | |
| Is there evidence of bus bunching? How frequent? | |
| Does a late trip cause the next trip to be late (cascading)? | |
| Where is data missing? Is missingness random or systematic? | |
| Which features show the strongest raw correlation with delay? | |

---
*Data Sources: TTC VISION GPS (AVL) · GTFS scheduled stop times · TransSee stop-level data (Darwin O'Connor)*  
*Reference: Steve Munro — Methodology for Analysis of TTC Vehicle Tracking Data (stevemunro.ca)*
